# NOTEBOOK 02: *FEATURE ENGINEERING*

In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle('data/pickles/df_analisis.pkl')

In [3]:
df.shape

(8323770, 19)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8323770 entries, 0 to 8323769
Data columns (total 19 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   id                           object 
 1   item                         object 
 2   category                     object 
 3   department                   object 
 4   store                        object 
 5   store_code                   object 
 6   region                       object 
 7   yearweek                     object 
 8   units_sold                   int64  
 9   sell_price                   float64
 10  year                         int32  
 11  quarter                      int32  
 12  event                        object 
 13  producto_decreciente_ny      bool   
 14  producto_creciente_ny        bool   
 15  producto_decreciente_boston  bool   
 16  producto_creciente_boston    bool   
 17  producto_decreciente_phi     bool   
 18  producto_creciente_phi       bool   
dtype

In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,8323770,30490,SUPERMARKET_3_827_PHI_3,273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item,8323770,3049,SUPERMARKET_3_827,2730,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,8323770,3,SUPERMARKET,3923010,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department,8323770,7,SUPERMARKET_3,2246790,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store,8323770,10,Greenwich_Village,832377,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_code,8323770,10,NYC_1,832377,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,8323770,3,New York,3329508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
yearweek,8323770,273,201616,30490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
units_sold,8323770.0,NaN,NaN,NaN,7.870148,23.693227,0.0,0.0,2.0,7.0,3976.0
sell_price,8323770.0,NaN,NaN,NaN,5.418773,4.442807,0.012,2.566991,4.034775,7.025,134.15


In [6]:
# Eliminamos la columna store_code ya que es redundante con store

df.drop(columns=['store_code'], inplace=True)

In [7]:
# Creamos un unique_id para cada combinación de id+semana

df['unique_id'] = df['id'].astype(str) + "_" + df['yearweek'].astype(str)

# EVENTS mejorado

In [8]:
df_event = pd.read_csv('data\eventos_mejorados_ohe.csv', index_col=0)

<>:1: SyntaxWarning: invalid escape sequence '\e'
<>:1: SyntaxWarning: invalid escape sequence '\e'
C:\Users\jorda\AppData\Local\Temp\ipykernel_10636\774843096.py:1: SyntaxWarning: invalid escape sequence '\e'
  df_event = pd.read_csv('data\eventos_mejorados_ohe.csv', index_col=0)


In [9]:
df_event['yearweek'] = df_event['yearweek'].astype(str)

In [10]:
print(f'Shape de df antes del merge: {df.shape}')
print(f'Shape de df_event antes del merge: {df_event.shape}\n')
print(f'El shape final debería ser: ({df.shape[0]},{df.shape[1]+df_event.shape[1]-1})\n')

df = df.merge(df_event, on=['yearweek'], how="left")

print(f'Shape después del merge: {df.shape}')

Shape de df antes del merge: (8323770, 19)
Shape de df_event antes del merge: (112, 28)

El shape final debería ser: (8323770,46)

Shape después del merge: (8323770, 46)


In [11]:
df.isnull().sum()

id                                   0
item                                 0
category                             0
department                           0
store                                0
region                               0
yearweek                             0
units_sold                           0
sell_price                           0
year                                 0
quarter                              0
event                                0
producto_decreciente_ny              0
producto_creciente_ny                0
producto_decreciente_boston          0
producto_creciente_boston            0
producto_decreciente_phi             0
producto_creciente_phi               0
unique_id                            0
event_Buddhist                 5305260
event_ChineeseNewYear          5305260
event_Christmas                5305260
event_Columbus                 5305260
event_Hindu                    5305260
event_Independence             5305260
event_JewNewYear         

In [12]:
df.fillna(0, inplace=True)

In [13]:
df.isnull().sum()

id                             0
item                           0
category                       0
department                     0
store                          0
region                         0
yearweek                       0
units_sold                     0
sell_price                     0
year                           0
quarter                        0
event                          0
producto_decreciente_ny        0
producto_creciente_ny          0
producto_decreciente_boston    0
producto_creciente_boston      0
producto_decreciente_phi       0
producto_creciente_phi         0
unique_id                      0
event_Buddhist                 0
event_ChineeseNewYear          0
event_Christmas                0
event_Columbus                 0
event_Hindu                    0
event_Independence             0
event_JewNewYear               0
event_Juneteenth               0
event_Labour                   0
event_MartinLutherKing         0
event_Memorial                 0
event_NewY

In [14]:
# Eliminamos la columna 'event'
df.drop('event',axis=1,inplace=True)

# TEMPORALES

## date, year, month, week y day

In [15]:
df['yearweek'] = df['yearweek'].astype(str)  # Aseguramos que sea string
df['year'] = df['yearweek'].str[:4].astype(int)  # Extraemos el año
df['week'] = df['yearweek'].str[4:].astype(int)  # Extraemos la semana

# Convertimos a datetime (estableciendo el primer día de la semana)
df['date'] = pd.to_datetime(df['year'].astype(str) + df['week'].astype(str) + '1', format='%Y%W%w')

df['month'] = df['date'].dt.month # Extraemos el mes

## season

In [16]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'

df['season'] = df['month'].apply(get_season)

# METEOROLÓGICAS

In [17]:
df_meteo = pd.read_csv('data/meteo_semanal.csv', index_col=0)

In [18]:
df_meteo.head()

,region,yearweek,max_wind,min_wind,avg_wind,max_rain,min_rain,avg_rain,max_snow,min_snow,avg_snow,max_tempmax,min_tempmax,avg_tempmax,max_tempmin,min_tempmin,avg_tempmin
0,Boston,201052,3.1,2.7,2.900000,3.8,0.0,1.900000,0.0,0.0,0.000000,13.3,10.6,11.950000,3.3,1.7,2.500000
1,Boston,201101,7.5,2.8,5.171429,4.1,0.0,1.128571,48.0,0.0,13.857143,3.9,0.0,2.157143,-1.7,-5.6,-3.585714
2,Boston,201102,10.6,2.9,6.114286,35.6,0.0,5.085714,371.0,0.0,53.000000,1.7,-3.9,0.085714,-2.2,-9.4,-6.100000
3,Boston,201103,6.3,3.6,4.557143,24.1,0.0,6.771429,185.0,0.0,33.285714,4.4,-6.1,-0.942857,0.0,-15.0,-7.142857
4,Boston,201104,6.5,2.3,3.871429,17.3,0.0,3.057143,224.0,0.0,38.857143,2.8,-10.6,-0.314286,-3.3,-18.9,-8.314286


In [19]:
df_meteo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 942 entries, 0 to 941
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   region       942 non-null    object 
 1   yearweek     942 non-null    int64  
 2   max_wind     942 non-null    float64
 3   min_wind     942 non-null    float64
 4   avg_wind     942 non-null    float64
 5   max_rain     942 non-null    float64
 6   min_rain     942 non-null    float64
 7   avg_rain     942 non-null    float64
 8   max_snow     942 non-null    float64
 9   min_snow     942 non-null    float64
 10  avg_snow     942 non-null    float64
 11  max_tempmax  942 non-null    float64
 12  min_tempmax  942 non-null    float64
 13  avg_tempmax  942 non-null    float64
 14  max_tempmin  942 non-null    float64
 15  min_tempmin  942 non-null    float64
 16  avg_tempmin  942 non-null    float64
dtypes: float64(15), int64(1), object(1)
memory usage: 132.5+ KB


In [20]:
df_meteo['yearweek'] = df_meteo['yearweek'].astype(str)

In [21]:
# Hacemos el merge con una comprobación de seguridad
print(f'Shape de df antes del merge: {df.shape}')
print(f'Shape de df_meteo antes del merge: {df_meteo.shape}\n')
print(f'El shape final debería ser: ({df.shape[0]},{df.shape[1]+df_meteo.shape[1]-2})\n')

df = df.merge(df_meteo, on=['yearweek', 'region'], how="left")

print(f'Shape después del merge: {df.shape}')

Shape de df antes del merge: (8323770, 49)
Shape de df_meteo antes del merge: (942, 17)

El shape final debería ser: (8323770,64)

Shape después del merge: (8323770, 64)


In [22]:
df.head()

,id,item,category,department,store,region,yearweek,units_sold,sell_price,year,...,avg_rain,max_snow,min_snow,avg_snow,max_tempmax,min_tempmax,avg_tempmax,max_tempmin,min_tempmin,avg_tempmin
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,New York,201105,0,10.770815,2011,...,3.914286,28.0,0.0,5.857143,7.2,-1.1,2.442857,2.8,-5.6,-2.785714
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,New York,201105,0,4.525800,2011,...,3.914286,28.0,0.0,5.857143,7.2,-1.1,2.442857,2.8,-5.6,-2.785714
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,New York,201105,0,3.739772,2011,...,3.914286,28.0,0.0,5.857143,7.2,-1.1,2.442857,2.8,-5.6,-2.785714
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,New York,201105,0,5.955215,2011,...,3.914286,28.0,0.0,5.857143,7.2,-1.1,2.442857,2.8,-5.6,-2.785714
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,New York,201105,0,3.165741,2011,...,3.914286,28.0,0.0,5.857143,7.2,-1.1,2.442857,2.8,-5.6,-2.785714


### temp_avg_range

In [23]:
df['temp_avg_range'] = df['avg_tempmax'] - df['avg_tempmin']

### has_rain y has_snow

Permite inferir la presencia de lluvia o nieve y separarla de la cantidad de estas

In [24]:
df['has_rain'] = df['max_rain'] > 0
df['has_snow'] = df['max_snow'] > 0

In [25]:
for ciudad in df.region.unique():
    df_ciudad = df[df['region'] == ciudad]
    
    semanas_total = df_ciudad['yearweek'].nunique()
    
    # Semanas en las que no paró de llover (llovió toda la semana)
    semanas_lluvia = df_ciudad[df_ciudad['min_rain'] != 0]['yearweek'].nunique()
    
    # Semanas en las que no paró de nevar (nevo toda la semana)
    semanas_nieve = df_ciudad[df_ciudad['min_snow'] != 0]['yearweek'].nunique()
    
    print(f'En {ciudad} hubo {semanas_lluvia} semanas de {semanas_total} en las que no paró de llover\n')
    print(f'En {ciudad} hubo {semanas_nieve} semanas de {semanas_total} en las que no paró de nevar\n')


En New York hubo 0 semanas de 273 en las que no paró de llover

En New York hubo 0 semanas de 273 en las que no paró de nevar

En Boston hubo 1 semanas de 273 en las que no paró de llover

En Boston hubo 0 semanas de 273 en las que no paró de nevar

En Philadelphia hubo 0 semanas de 273 en las que no paró de llover

En Philadelphia hubo 0 semanas de 273 en las que no paró de nevar



In [26]:
for ciudad in df.region.unique():
    df_ciudad = df[df['region'] == ciudad]
    
    semanas_total = df_ciudad['yearweek'].nunique()
    
    # Semanas en las que no llovió en ningún momento de la semana (max_rain == 0)
    semanas_lluvia = df_ciudad[df_ciudad['max_rain'] == 0]['yearweek'].nunique()
    
    # Semanas en las que no nevó en ningún momento de la semana (max_snow == 0)
    semanas_nieve = df_ciudad[df_ciudad['max_snow'] == 0]['yearweek'].nunique()
    
    print(f'En {ciudad} hubo {semanas_lluvia} semanas de {semanas_total} en las que no llovió\n')
    print(f'En {ciudad} hubo {semanas_nieve} semanas de {semanas_total} en las que no nevó\n')



En New York hubo 24 semanas de 273 en las que no llovió

En New York hubo 230 semanas de 273 en las que no nevó

En Boston hubo 21 semanas de 273 en las que no llovió

En Boston hubo 209 semanas de 273 en las que no nevó

En Philadelphia hubo 21 semanas de 273 en las que no llovió

En Philadelphia hubo 228 semanas de 273 en las que no nevó



Dada la utilidad de las variables min_rain y min_snow, se decide eliminarlas

In [27]:
df.drop(['min_rain','min_snow'], axis=1, inplace=True)

### extreme_weather
Se calcula extreme weather como el 5% de peores lluvias, nieve o viento de cada ciudad. Esto se hace porque no se perciben de la misma manera 10mm de lluvia en Murcia que en Vigo

In [28]:
# Calculamos los percentiles por región
rain_thresh = df.groupby('region')['avg_rain'].transform(lambda x: x.quantile(0.95))
snow_thresh = df.groupby('region')['avg_snow'].transform(lambda x: x.quantile(0.95))
wind_thresh = df.groupby('region')['avg_wind'].transform(lambda x: x.quantile(0.95))

# Clasificamos como clima extremo si sobrepasa el umbral correspondiente de su región
df['extreme_weather'] = (
    (df['avg_rain'] > rain_thresh) |
    (df['avg_snow'] > snow_thresh) |
    (df['avg_wind'] > wind_thresh)
).astype(int)


# PRECIO

### price_diff_from_median_item
Diferencia de precio respecto a la mediana de venta de ese producto

In [29]:
df['median_price_item'] = df.groupby('item')['sell_price'].transform('median')
df['price_diff_from_median_item'] = df['sell_price'] - df['median_price_item']
df.drop('median_price_item', axis=1, inplace=True)

### price_region_week_avg y price_per_region_avg_diff

Permite identificar precios locales desalineados con la media regional

In [30]:
df['price_region_week_avg'] = df.groupby(['region', 'yearweek'])['sell_price'].transform('mean')
df['price_per_region_avg_diff'] = df['sell_price'] - df['price_region_week_avg']

### income
Asigna los ingresos obtenidos esa semana gracias a ese producto en esa tienda

In [31]:
df['income'] = df['units_sold'] * df['sell_price']

# TIME SERIES

Lags de ventas

In [32]:
df['lag_1'] = df.groupby('id')['units_sold'].shift(1) # ventas de hace 1 semana
df['lag_2'] = df.groupby('id')['units_sold'].shift(2) # ventas de hace 2 semanas
df['lag_3'] = df.groupby('id')['units_sold'].shift(3) # ventas de hace 3 semanas

Diff

In [33]:
df['diff_lag_1'] = df.groupby('id')['units_sold'].shift(1).diff(1)
df['diff_lag_2'] = df.groupby('id')['units_sold'].shift(2).diff(2)
df['diff_lag_3'] = df.groupby('id')['units_sold'].shift(3).diff(3)

Media y Desviación móvil


In [34]:
# ventana de 3 semanas
df['rolling_mean_3'] = df.groupby('id')['units_sold'].rolling(window=3).mean().reset_index(level=0, drop=True) 
df['rolling_std_3'] = df.groupby('id')['units_sold'].rolling(window=3).std().reset_index(level=0, drop=True)

Diferencia en precio respecto a la semana anterior

In [35]:
df['price_change'] = df.groupby('id')['sell_price'].diff() 

Promedio de ventas por categoría

In [36]:
df['mean_units_by_category'] = df.groupby('category')['units_sold'].transform('mean')

Eliminamos las filas correspondientes a las 3 primeras semanas (porque presentan nulos en el lag)

In [37]:
df.dropna(inplace=True)

In [38]:
df.isnull().sum()

id                        0
item                      0
category                  0
department                0
store                     0
                         ..
diff_lag_3                0
rolling_mean_3            0
rolling_std_3             0
price_change              0
mean_units_by_category    0
Length: 80, dtype: int64

In [39]:
df.columns

Index(['id', 'item', 'category', 'department', 'store', 'region', 'yearweek',
       'units_sold', 'sell_price', 'year', 'quarter',
       'producto_decreciente_ny', 'producto_creciente_ny',
       'producto_decreciente_boston', 'producto_creciente_boston',
       'producto_decreciente_phi', 'producto_creciente_phi', 'unique_id',
       'event_Buddhist', 'event_ChineeseNewYear', 'event_Christmas',
       'event_Columbus', 'event_Hindu', 'event_Independence',
       'event_JewNewYear', 'event_Juneteenth', 'event_Labour',
       'event_MartinLutherKing', 'event_Memorial', 'event_NewYear',
       'event_President', 'event_RamadanEnds', 'event_RamadanStarts',
       'event_StPatrick', 'event_Superbowl', 'event_Thanksgiving',
       'event_Veterans', 'event_YomKipur', 'event_group_buddhist',
       'event_group_chinese', 'event_group_hindu', 'event_group_islam',
       'event_group_jew', 'event_group_national', 'has_event', 'week', 'date',
       'month', 'season', 'max_wind', 'min_wind', '

# CSV para PowerBI

In [40]:
# df.to_csv('data/dashboard/df_features.csv', index=False, sep=';', decimal=',')

# PICKLE

In [41]:
df.to_pickle('data/pickles/df_features.pkl')